In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
def check_system_info():
    """Display system information for debugging"""
    print("=" * 60)
    print("SYSTEM INFORMATION")
    print("=" * 60)
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"CUDA Version: {torch.version.cuda}")
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    else:
        print("Running on CPU (slower performance)")
    print("=" * 60)

check_system_info()

SYSTEM INFORMATION
PyTorch Version: 2.10.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Device: Tesla T4
GPU Memory: 14.56 GB


In [3]:
# Configuration - Change this based on your hardware
USE_QUANTIZATION = '4bit'  # Options: None, '8bit', '4bit'

print(f"Configuration: Quantization = {USE_QUANTIZATION if USE_QUANTIZATION else 'None (Full Precision)'}")

Configuration: Quantization = 4bit


In [4]:
from transformers import BitsAndBytesConfig

# Configure model loading based on quantization
model_kwargs = {
    "trust_remote_code": True,
    "device_map": "auto"
}

In [5]:
def load_model(use_quantization=None):
    """
    Load Qwen 2.5 7B Instruct model

    Args:
        use_quantization: None, '8bit', or '4bit'
    """
    model_name = "Qwen/Qwen2.5-7B-Instruct"

    print(f"Loading model: {model_name}")
    print(f"Quantization: {use_quantization if use_quantization else 'None (FP16/BF16)'}")
    print("This may take a few minutes on first run...")
    print()

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    # Configure model loading based on quantization
    model_kwargs = {
        "trust_remote_code": True,
        "device_map": "auto"  # Automatically distribute across available devices
    }

    if use_quantization == '8bit':
        print("Loading with 8-bit quantization (requires bitsandbytes)")
        model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif use_quantization == '4bit':
        print("Loading with 4-bit quantization (requires bitsandbytes)")
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,   # faster compute
            bnb_4bit_use_double_quant=True,            # saves a bit more VRAM
            bnb_4bit_quant_type="nf4"                  # NF4 is standard for QLoRA/inference
        )
    else:
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            model_kwargs["torch_dtype"] = torch.bfloat16
            print("Using BF16 precision")
        else:
            model_kwargs["torch_dtype"] = torch.float16
            print("Using FP16 precision")

    # Load model
    # pip install -U bitsandbytes>=0.46.1, for colab environment
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        **model_kwargs
    )

    print("Model loaded successfully!")

    return model, tokenizer

# Load the model
model, tokenizer = load_model(use_quantization=USE_QUANTIZATION)

Loading model: Qwen/Qwen2.5-7B-Instruct
Quantization: 4bit
This may take a few minutes on first run...



Loading with 4-bit quantization (requires bitsandbytes)


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
def generate_response(model, tokenizer, prompt, max_new_tokens=512, temperature=0.7):
    """Generate a response from the model"""

    # Format the prompt using chat template
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    model_inputs = tokenizer([text], return_tensors="pt")

    # Move to same device as model
    if torch.cuda.is_available():
        model_inputs = model_inputs.to(model.device)

    # Generate
    print("Generating response...")
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode only the generated tokens (exclude the prompt)
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

print("Generation function defined successfully!")

Generation function defined successfully!


In [7]:
# Example 1: Fibonacci sequence
prompt1 = "Introduce the linear regression."
# prompt1 = 'Write a python function to print hello world'
print("=" * 60)
print(f"Prompt: {prompt1}")
print("=" * 60)

response1 = generate_response(
    model,
    tokenizer,
    prompt1,
    max_new_tokens=512,
    temperature=0.7
)

print("\nResponse:")
print(response1)

Prompt: Introduce the linear regression.
Generating response...

Response:
Linear regression is a fundamental statistical and machine learning technique used to model the relationship between a dependent variable (often denoted as \(Y\)) and one or more independent variables (often denoted as \(X\)). It is widely used in various fields such as economics, finance, social sciences, and data science for prediction, forecasting, and understanding relationships between variables.

### Basic Concepts

1. **Simple Linear Regression**: This involves modeling the relationship between a single independent variable (\(X\)) and a dependent variable (\(Y\)). The basic form of the equation is:
   \[
   Y = \beta_0 + \beta_1 X + \epsilon
   \]
   - \(Y\) is the dependent variable.
   - \(X\) is the independent variable.
   - \(\beta_0\) is the intercept, representing the value of \(Y\) when \(X = 0\).
   - \(\beta_1\) is the slope, indicating how much \(Y\) changes with respect to \(X\).
   - \(\epsi

In [8]:
# Clear GPU memory
import gc

# Delete model and tokenizer
del model
del tokenizer

# Run garbage collection
gc.collect()

# Clear CUDA cache if using GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared")

print("Memory freed successfully!")

GPU memory cleared
Memory freed successfully!
